# FER-CE — Démo complète (Google Colab)

**Reconnaissance des Expressions Faciales Composées** sur RAF-DB (11 classes, numérotées 1..11).

Ce notebook exécute **tout le projet** de bout en bout :

| Étape | Couche | Ce qui se passe |
|------:|:------:|-----------------|
| 1 |  —  | Préparation Colab : GPU, Drive, dézippage, dépendances |
| 2 |  1  | Audit du dataset + cache des visages (sans utiliser de nom d'émotion) |
| 3 |  2  | Entraînement multi-modèles : ResNet-50, ViT, Swin |
| 4 |  2  | Évaluation et **comparaison** des modèles |
| 5 |  2  | Inférence sur des images individuelles + top-3 |
| 6 |  2  | **Vision-LLM** Qwen2-VL : explication zero-shot des émotions |
| 7 |  3  | **Grad-CAM** + cohérence image/texte (Causal Emotion Grounding) |

> **Important — sans labels sémantiques.** Tout le pipeline traite les classes par leur **numéro** (1..11), jamais par leur nom d'émotion. C'est volontaire — l'entraînement et l'évaluation n'ont besoin d'aucune sémantique : seul le **Vision-LLM** parle d'émotions, en zero-shot.

Toute la configuration se trouve dans `config.py`. Voir aussi `explanation.md`.

## 1.  Préparation de l'environnement

In [ ]:
# 1.1  GPU disponible ?
!nvidia-smi -L || echo "⚠ Aucun GPU détecté — passe en CPU (l'entraînement sera lent)."

In [ ]:
# 1.2  Monte Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.3  Chemin vers le zip du projet sur Drive
#
# Le zip `deep_learning.zip` est plat : il contient les fichiers .py, le
# notebook, requirements.txt, et 3 sous-zips (Image.zip, Annotation.zip,
# EmoLabel.zip). Aucun dossier imbriqué.
DRIVE_ZIP_PATH = "/content/drive/MyDrive/deep_learning.zip"   # <-- adapte si besoin
PROJECT_DIR     = "/content/deep_learning"

In [ ]:
# 1.4  Dézippage du projet et des données
import os, shutil, zipfile

if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH) as zf:
    zf.extractall(PROJECT_DIR)

# Sous-zips de données -> on les dézippe à côté des scripts
for sub in ("Image.zip", "Annotation.zip", "EmoLabel.zip"):
    path = os.path.join(PROJECT_DIR, sub)
    if os.path.exists(path):
        with zipfile.ZipFile(path) as zf:
            zf.extractall(PROJECT_DIR)
        os.remove(path)
        print(f"  [✓] {sub} dézippé")

# Cas Image/Image/original/... (un seul niveau de trop) -> on aplatit.
for name in ("Image", "Annotation", "EmoLabel"):
    outer = os.path.join(PROJECT_DIR, name)
    inner = os.path.join(outer, name)
    if os.path.isdir(inner):
        for child in os.listdir(inner):
            shutil.move(os.path.join(inner, child), os.path.join(outer, child))
        os.rmdir(inner)

print("\nContenu de", PROJECT_DIR)
for entry in sorted(os.listdir(PROJECT_DIR)):
    full = os.path.join(PROJECT_DIR, entry)
    tag  = "DIR" if os.path.isdir(full) else "   "
    print(f"  [{tag}] {entry}")

os.chdir(PROJECT_DIR)
print("\ncwd =", os.getcwd())

In [ ]:
# 1.5  Dépendances (le runtime Colab fournit déjà torch/torchvision/sklearn).
#       Pillow<12 -> compatibilité Qwen2-VL ; les autres sont indispensables.
!pip install -q -U transformers accelerate bitsandbytes timm seaborn "pillow<12"

In [ ]:
# 1.6  Tous les scripts s'importent ?
import importlib, sys
sys.path.insert(0, PROJECT_DIR)

for m in ["config", "training_utils", "model", "dataset",
         "face_detection", "data_exploration", "check_dataset", "preprocessing",
         "train", "evaluate", "vision_llm", "explain", "interpret",
         "main_test_model", "main"]:
    importlib.import_module(m)
    print(f"  [OK] {m}")

import config
print("\nConfig clé :")
print(f"  NUM_CLASSES = {config.NUM_CLASSES}  (numéros 1..{config.NUM_CLASSES})")
print(f"  MODELS      = {config.MODELS}")
print(f"  EPOCHS      = {config.EPOCHS}  BATCH = {config.BATCH_SIZE}")
print(f"  USE_MIXUP={config.USE_MIXUP}  USE_CUTMIX={config.USE_CUTMIX}  USE_EMA={config.USE_EMA}")

## 2.  Couche 1 — Audit du dataset (sans labels sémantiques)

On regarde la **répartition des numéros de classes** (1..11) et on vérifie l'intégrité des images.  
Aucune sémantique n'est utilisée : chaque image est identifiée par son numéro de classe — l'esprit du *self-supervised learning* pour la partie reconnaissance.

In [ ]:
import data_exploration; data_exploration.main()

In [ ]:
import check_dataset; check_dataset.main()

In [ ]:
# 2.1  Construit le cache des visages recadrés (rapide après la 1re exécution).
from dataset import FERDataset
train_ds = FERDataset("train", train=True)
test_ds  = FERDataset("test",  train=False)
print(f"  train = {len(train_ds)} images, test = {len(test_ds)} images")
print(f"  classes train -> {train_ds.class_counts().tolist()}")

In [ ]:
# 2.2  Aperçu visuel : distribution + échantillons par numéro de classe.
from IPython.display import Image
for f in ("class_distribution.png", "sample_images.png"):
    path = os.path.join(config.OUTPUT_DIR, f)
    if os.path.exists(path):
        display(Image(filename=path))

## 3.  Couche 2 — Entraînement multi-modèles

On entraîne plusieurs architectures avec le même pipeline (MixUp + CutMix + EMA + label smoothing + cosine LR + AMP). Tout est piloté par `config.py`.

> **Astuce GPU.** Sur T4 (Colab gratuit), 45 époques × 3 modèles = ~2h. Si tu veux juste tester, passe `EPOCHS = 10` ci-dessous.

In [ ]:
import config, train, importlib

# Optionnel : surcharges rapides pour tester.
# config.EPOCHS = 10
# config.BATCH_SIZE = 64
# config.USE_FOCAL = True

importlib.reload(train)
histories = {}
for model_name in config.MODELS:
    print(f"\n\n##########  {model_name}  ##########")
    histories[model_name] = train.train_model(model_name, epochs=config.EPOCHS)

## 4.  Couche 2 — Évaluation et comparaison

Pour chaque modèle :
* accuracy globale train vs test ;
* accuracy par classe (numéros 1..11) ;
* matrice de confusion ;
* `metrics_<model>.json` enregistré dans `outputs/`.

In [ ]:
import evaluate, importlib
importlib.reload(evaluate)
all_metrics = {}
for model_name in config.MODELS:
    print(f"\n--- Évaluation : {model_name} ---")
    try:
        all_metrics[model_name] = evaluate.evaluate_model(model_name,
                                                        prefer_ema=True,
                                                        use_tta=config.TTA)
    except FileNotFoundError as e:
        print(f"  (skip) {e}")

In [ ]:
# 4.1  Comparaison synthétique : tableau + graphique des accuracies test.
import matplotlib.pyplot as plt

print(f"  {'Modèle':<32s} {'Acc train':>10s} {'Acc test':>10s} {'F1 test':>10s}")
print("  " + "-" * 66)
for name, m in all_metrics.items():
    print(f"  {name:<32s} {m['accuracy']['train']*100:>9.2f}%"
          f" {m['accuracy']['test']*100:>9.2f}% {m['macro_f1']['test']*100:>9.2f}%")

if all_metrics:
    names = list(all_metrics.keys())
    acc   = [all_metrics[n]['accuracy']['test'] for n in names]
    f1    = [all_metrics[n]['macro_f1']['test']  for n in names]
    x = range(len(names))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([i - 0.18 for i in x], acc, 0.36, label="Accuracy test", color="#4C9BE8")
    ax.bar([i + 0.18 for i in x], f1,  0.36, label="Macro-F1 test", color="#E8734C")
    ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=15)
    ax.set_ylim(0, 1); ax.set_ylabel("Score")
    ax.set_title("Comparaison des modèles — RAF-DB (11 classes)", fontweight="bold")
    ax.grid(axis='y', alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

## 5.  Couche 2 — Inférence sur des images individuelles

Pour chaque image, on affiche : image originale · visage recadré · top-3 des prédictions (numéros de classe + confiance).

In [ ]:
import random, os, main_test_model, importlib
importlib.reload(main_test_model)

BEST_MODEL = max(all_metrics, key=lambda n: all_metrics[n]['accuracy']['test']) \
             if all_metrics else config.DEFAULT_MODEL
print(f"Meilleur modèle: {BEST_MODEL}")

model = main_test_model.load_classifier(BEST_MODEL, prefer_ema=True)
imgs  = [f for f in os.listdir(config.IMAGE_DIR) if f.startswith("test")][:6]
random.seed(0)
for i, name in enumerate(random.sample(imgs, min(6, len(imgs)))):
    main_test_model.test_one_image(os.path.join(config.IMAGE_DIR, name),
                                   model, mode="bbox", index=i, show=True)

## 6.  Couche 2 — Vision-LLM (Qwen2-VL, zero-shot)

Un modèle vision-langage **non entraîné** sur cette tâche décrit l'émotion composée du visage et **explique** les indices faciaux (sourcils, yeux, bouche, etc.).

Le numéro de classe vient du classifieur ; l'explication vient du Vision-LLM.

In [ ]:
import explain, importlib
importlib.reload(explain)

sample = [os.path.join(config.IMAGE_DIR, f)
          for f in os.listdir(config.IMAGE_DIR) if f.startswith("test")][:5]
explanations = explain.explain_images(sample, model_name=BEST_MODEL)

In [ ]:
# Aperçu des images générées (`outputs/explain_<i>.png`).
from IPython.display import Image as IPImage
for i in range(len(explanations)):
    p = os.path.join(config.OUTPUT_DIR, f"explain_{i}.png")
    if os.path.exists(p): display(IPImage(filename=p))

## 7.  Couche 3 — Interprétation : Grad-CAM + cohérence Vision-LLM

* **Grad-CAM** : où le CNN regarde dans le visage pour prendre sa décision.
* **Découpage en 3 bandes** (front/sourcils, yeux, bouche/joues) : approximation des *Action Units*.
* **Cohérence** *Causal Emotion Grounding* : la zone activée par le CNN est-elle celle citée par le Vision-LLM ?

In [ ]:
import interpret, importlib
importlib.reload(interpret)

# Avec --vlm pour la cohérence image/texte.
interp = interpret.interpret_images(sample, model_name=BEST_MODEL,
                                    use_vlm=True, prefer_ema=True)

In [ ]:
# Aperçu (`outputs/interpret_<i>.png`)
from IPython.display import Image as IPImage
for i in range(len(interp)):
    p = os.path.join(config.OUTPUT_DIR, f"interpret_{i}.png")
    if os.path.exists(p): display(IPImage(filename=p))

## 8.  Conclusion

**Récapitulatif :**

* Couche 1 — préparation **sans étiquette sémantique** (numéros 1..11) : audit, recadrage, augmentation.
* Couche 2 — entraînement **haute efficacité** (MixUp + CutMix + EMA + label smoothing + AMP) sur 3 architectures, comparées sur les mêmes données.
* Couche 2 — **Vision-LLM** Qwen2-VL en zero-shot pour expliquer textuellement les émotions.
* Couche 3 — **Grad-CAM + cohérence** : on vérifie que la zone activée par le CNN est bien celle mentionnée par le LLM (*Causal Emotion Grounding*).

Tous les résultats (PNG, JSON, .pth) sont dans `outputs/`.

Voir `explanation.md` pour une description fonction par fonction et concept par concept.

In [ ]:
# (Optionnel) Copier `outputs/` sur Drive pour conserver les modèles et figures.
# OUT_DRIVE = "/content/drive/MyDrive/fer_ce_outputs"
# import shutil, os
# os.makedirs(OUT_DRIVE, exist_ok=True)
# for f in os.listdir(config.OUTPUT_DIR):
#     shutil.copy(os.path.join(config.OUTPUT_DIR, f), OUT_DRIVE)
# print("  [✓]", OUT_DRIVE)